# Parcellation & SLIC Supervoxel Demo

Demonstrates the **anatomically-constrained 3D SLIC** algorithm (Eq. 5-6).

- **Eq. 5:** D*_k(v) = D_k(v) + lambda * Delta(A(v), A(s_k))
- **Eq. 6:** d_intensity = sqrt(sum_m w_m * (I^m_v - I^m_center)^2)
- ~8000 supervoxels per brain (target_supervoxel_size=256)
- numba JIT-accelerated inner loop for production performance

In [ ]:
%matplotlib inline

import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt

from stroke_gat.config import load_config
from stroke_gat.data.service import DataService
from stroke_gat.data.transforms import compute_brain_mask
from stroke_gat.slic.atlas_slic import AnatomicalSLIC
from stroke_gat.visualization.parcellation import plot_atlas_slices, plot_arterial_territories
from stroke_gat.visualization.supervoxels import (
    plot_supervoxel_slices,
    plot_supervoxel_parcellated_brain,
    plot_supervoxel_size_histogram,
    plot_atlas_adherence,
)

In [ ]:
# Load configuration and data for one subject
config = load_config("../configs/default.yaml")
service = DataService(config.paths)

subjects = service.discover_subjects_bids()
subject_id = subjects[0]
print(f"Working with subject: {subject_id}")

# Load modalities and atlas
modalities_nifti = service.load_subject_modalities(subject_id)
atlas_data, atlas_img = service.load_atlas()
atlas_labels = service.load_atlas_labels()

# Extract numpy arrays
modalities = {name: img.get_fdata(dtype=np.float32) for name, img in modalities_nifti.items()}

print(f"T1 shape: {modalities['T1'].shape}")
print(f"Atlas regions: {len(np.unique(atlas_data)) - 1}")

In [ ]:
# Visualize the atlas parcellation
fig = plot_atlas_slices(modalities["T1"], atlas_data, n_slices=6, alpha=0.35)
plt.show()

# Arterial territories with labels
fig = plot_arterial_territories(atlas_data, atlas_labels)
plt.show()

In [ ]:
# Run anatomically-constrained SLIC segmentation
print(f"SLIC parameters:")
print(f"  Target supervoxel size: {config.slic.target_supervoxel_size}")
print(f"  Compactness:           {config.slic.compactness}")
print(f"  Atlas penalty lambda:  {config.slic.atlas_penalty_lambda}")
print(f"  Max iterations:        {config.slic.max_iterations}")

slic = AnatomicalSLIC(config.slic)

# Compute brain mask from T1
brain_mask = compute_brain_mask(modalities["T1"])

print(f"\nRunning SLIC on {len(modalities)}-modal volume {modalities['T1'].shape}...")
supervoxel_labels = slic.segment(modalities, atlas_data, brain_mask=brain_mask)

n_supervoxels = len(np.unique(supervoxel_labels[supervoxel_labels >= 0]))
expected = brain_mask.sum() // config.slic.target_supervoxel_size
print(f"Generated {n_supervoxels} supervoxels (expected ~{expected:,})")

In [ ]:
# Plot supervoxel boundaries on axial slices
fig = plot_supervoxel_slices(modalities["T1"], supervoxel_labels, n_slices=6)
plt.show()

In [ ]:
# Combined view: atlas regions + supervoxel boundaries
fig = plot_supervoxel_parcellated_brain(
    modalities["T1"], supervoxel_labels, atlas_data
)
plt.show()

In [ ]:
# Supervoxel size distribution
fig = plot_supervoxel_size_histogram(
    supervoxel_labels,
    target_size=config.slic.target_supervoxel_size,
)
plt.show()

In [ ]:
# Atlas boundary adherence analysis
fig = plot_atlas_adherence(supervoxel_labels, atlas_data)
plt.show()

## Discussion

**Supervoxel quality metrics:**

- **Size uniformity:** Distribution centered around target_supervoxel_size (256 voxels).
  Variation is expected at atlas region boundaries.
- **Atlas adherence:** With lambda=0.5 penalty, nearly all supervoxels stay within
  a single arterial territory. This gives each graph node a clear anatomical identity.
- **Boundary quality:** Supervoxels follow both intensity gradients (tissue boundaries)
  and atlas boundaries, providing a good over-segmentation.

Next: See `03_graph_construction_demo.ipynb` for building graphs from supervoxels.